# Modelagem Preditiva com Árvore de Decisão (Regressão)
## Previsão do custo de seguro de saúde — *Medical Cost Personal Dataset*

---

## 1. Escolha e Entendimento do Dataset

O dataset escolhido foi o **Medical Cost Personal Dataset** ([Kaggle – mirichoi0218/insurance](https://www.kaggle.com/datasets/mirichoi0218/insurance)), originalmente publicado no livro *Machine Learning with R*, de Brett Lantz. Ele contém **1.338 registros** de beneficiários de um plano de saúde nos Estados Unidos, com as seguintes colunas:

| Coluna | Tipo | Descrição |
|---|---|---|
| `age` | numérica | Idade do beneficiário principal (18 a 64 anos) |
| `sex` | categórica | Sexo (`female` / `male`) |
| `bmi` | numérica | Índice de Massa Corporal (IMC, kg/m²) |
| `children` | numérica | Número de filhos/dependentes cobertos pelo plano |
| `smoker` | categórica | Se a pessoa é fumante (`yes` / `no`) |
| `region` | categórica | Região de residência nos EUA (`northeast`, `northwest`, `southeast`, `southwest`) |
| **`charges`** | **numérica contínua** | **Custo médico individual cobrado pelo seguro (US$) — variável alvo** |

**Objetivo do modelo:** prever o valor de `charges` (custo anual do seguro, em dólares) a partir das características do beneficiário. Como o alvo é uma variável **contínua**, trata-se de um problema de **regressão**, que resolveremos com o algoritmo `DecisionTreeRegressor` do scikit-learn.

Esse dataset é interessante para árvores de decisão porque mistura variáveis numéricas e categóricas e porque o custo do seguro depende de **interações** entre variáveis (por exemplo, ser fumante *e* ter IMC alto), algo que uma árvore consegue capturar naturalmente com regras do tipo "se… então…".

In [ ]:
# Importação das bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
RANDOM_STATE = 42

### 1.1 Carregamento dos dados

O arquivo é o mesmo disponibilizado no Kaggle. Para não exigir login/API do Kaggle no Colab, ele é lido diretamente do repositório público do livro no GitHub.

In [ ]:
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(URL)

# Alternativa: upload manual do arquivo baixado do Kaggle
# from google.colab import files
# files.upload()                      # selecione o insurance.csv
# df = pd.read_csv("insurance.csv")

print(f"Formato do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe()

In [ ]:
# Distribuição das variáveis categóricas
for col in ["sex", "smoker", "region"]:
    print(df[col].value_counts(), "\n")

### 1.2 Análise exploratória rápida da variável alvo

Antes de modelar, vale olhar como `charges` se distribui e como se relaciona com as principais variáveis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["charges"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribuição do custo do seguro (charges)")
axes[0].set_xlabel("charges (US$)")

sns.boxplot(data=df, x="smoker", y="charges", hue="smoker", legend=False, ax=axes[1], palette="Set2")
axes[1].set_title("Custo por condição de fumante")

sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.6, ax=axes[2], palette="Set1")
axes[2].set_title("Custo x Idade (cor = fumante)")

plt.tight_layout()
plt.show()

**Observações iniciais:**
- A distribuição de `charges` é **assimétrica à direita**: a maioria das pessoas paga menos de US$ 15 mil, mas há uma cauda longa de custos altos.
- Ser **fumante** eleva muito o custo: a mediana dos fumantes é várias vezes maior que a dos não fumantes.
- O custo cresce com a **idade**, e o gráfico de dispersão mostra "faixas" bem definidas, um padrão em degraus que se encaixa bem em uma árvore de decisão.

---
## 2. Preparação dos Dados

Etapas do tratamento:
1. **Dados faltantes:** verificar e, se houver, tratar (mediana para numéricas, moda para categóricas);
2. **Duplicatas:** remover linhas repetidas;
3. **Variáveis categóricas → numéricas:**
   - `sex` e `smoker` são binárias → mapeamento direto para 0/1;
   - `region` tem 4 categorias sem ordem → *One-Hot Encoding* (`pd.get_dummies`).

> Observação: árvores de decisão **não exigem normalização/padronização** das variáveis numéricas, pois as divisões são feitas por limiares (ex.: `age <= 42.5`) e não por distâncias.

In [ ]:
# 2.1 Verificação de dados faltantes
print("Valores faltantes por coluna:")
print(df.isnull().sum())

In [ ]:
# Tratamento genérico (só tem efeito se existirem valores faltantes)
num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(exclude="number").columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Total de valores faltantes após o tratamento:", df.isnull().sum().sum())

In [ ]:
# 2.2 Remoção de duplicatas
n_dup = df.duplicated().sum()
print(f"Linhas duplicadas encontradas: {n_dup}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Formato após remoção: {df.shape}")

In [ ]:
# 2.3 Transformação das variáveis categóricas em numéricas
df_model = df.copy()

# Variáveis binárias -> 0/1
df_model["sex"] = df_model["sex"].map({"female": 0, "male": 1})
df_model["smoker"] = df_model["smoker"].map({"no": 0, "yes": 1})

# Variável nominal com mais de 2 categorias -> One-Hot Encoding
df_model = pd.get_dummies(df_model, columns=["region"], drop_first=True, dtype=int)

df_model.head()

In [ ]:
# 2.4 Separação entre variáveis explicativas (X) e alvo (y)
X = df_model.drop(columns="charges")
y = df_model["charges"]

# 2.5 Divisão treino (80%) / teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print("Variáveis usadas:", list(X.columns))

---
## 3. Treinamento do Modelo

Vamos treinar o `DecisionTreeRegressor` em duas etapas:

1. **Árvore sem restrições** (parâmetros padrão): cresce até cada folha ficar "pura". Serve de referência e mostra o problema de **overfitting** (sobreajuste).
2. **Árvore ajustada**: usamos `GridSearchCV` (validação cruzada com 5 *folds*) para escolher a profundidade máxima (`max_depth`) e o número mínimo de amostras por folha (`min_samples_leaf`), que são os principais hiperparâmetros para controlar a complexidade da árvore.

In [ ]:
# Função auxiliar para calcular as métricas de regressão
def avaliar(modelo, X_tr, y_tr, X_te, y_te, nome):
    resultados = []
    for conjunto, Xs, ys in [("Treino", X_tr, y_tr), ("Teste", X_te, y_te)]:
        pred = modelo.predict(Xs)
        resultados.append({
            "Modelo": nome,
            "Conjunto": conjunto,
            "MAE (US$)": mean_absolute_error(ys, pred),
            "RMSE (US$)": np.sqrt(mean_squared_error(ys, pred)),
            "R²": r2_score(ys, pred),
        })
    return pd.DataFrame(resultados)

In [ ]:
# 3.1 Árvore sem restrições (baseline)
arvore_livre = DecisionTreeRegressor(random_state=RANDOM_STATE)
arvore_livre.fit(X_train, y_train)

print(f"Profundidade da árvore: {arvore_livre.get_depth()}")
print(f"Número de folhas: {arvore_livre.get_n_leaves()}")
res_livre = avaliar(arvore_livre, X_train, y_train, X_test, y_test, "Árvore sem restrições")
res_livre

A árvore sem restrições "decora" o conjunto de treino (R² ≈ 1,0 no treino), mas o desempenho cai no teste: é o **overfitting** típico de árvores profundas. Ela criou centenas de folhas, muitas com uma única pessoa. A seguir, buscamos uma árvore mais simples que **generalize melhor**.

In [ ]:
# 3.2 Busca dos melhores hiperparâmetros com validação cruzada
param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": [1, 5, 10, 20, 30, 50],
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Melhores hiperparâmetros:", grid.best_params_)
print(f"R² médio na validação cruzada: {grid.best_score_:.4f}")

In [ ]:
# Como o R² de validação varia com a profundidade (melhor min_samples_leaf para cada profundidade)
cv_res = pd.DataFrame(grid.cv_results_)
cv_res["max_depth"] = cv_res["param_max_depth"].astype(str).replace("None", "Sem limite")
curva = cv_res.groupby("max_depth", sort=False)["mean_test_score"].max()

plt.figure(figsize=(8, 4))
plt.plot(curva.index, curva.values, marker="o")
plt.title("R² médio (validação cruzada) por profundidade máxima")
plt.xlabel("max_depth")
plt.ylabel("R² (CV)")
plt.show()

In [ ]:
# 3.3 Modelo final: DecisionTreeRegressor com os hiperparâmetros escolhidos
modelo = DecisionTreeRegressor(**grid.best_params_, random_state=RANDOM_STATE)
modelo.fit(X_train, y_train)

print(f"Profundidade da árvore final: {modelo.get_depth()}")
print(f"Número de folhas: {modelo.get_n_leaves()}")

---
## 4. Avaliação do Modelo

Métricas utilizadas:
- **MAE** (*Mean Absolute Error*): erro médio absoluto, em dólares. Indica, em média, quanto a previsão erra para mais ou para menos.
- **RMSE** (*Root Mean Squared Error*): raiz do erro quadrático médio, também em dólares. Penaliza mais os erros grandes.
- **R²** (coeficiente de determinação): proporção da variância do custo explicada pelo modelo (1,0 = perfeito; 0 = equivalente a prever sempre a média).

In [ ]:
res_final = avaliar(modelo, X_train, y_train, X_test, y_test, "Árvore ajustada")
comparacao = pd.concat([res_livre, res_final], ignore_index=True)
comparacao

In [ ]:
# Referência: um "modelo" ingênuo que sempre prevê a média do treino
pred_media = np.full(len(y_test), y_train.mean())
print(f"Baseline (prever sempre a média) -> MAE: US$ {mean_absolute_error(y_test, pred_media):,.2f} | "
      f"R²: {r2_score(y_test, pred_media):.4f}")
print(f"Custo médio no conjunto de teste: US$ {y_test.mean():,.2f}")

In [ ]:
# Valores reais x previstos no conjunto de teste
y_pred = modelo.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(y_test, y_pred, alpha=0.6, c=X_test["smoker"], cmap="coolwarm")
lim = [0, max(y_test.max(), y_pred.max()) * 1.05]
axes[0].plot(lim, lim, "k--", label="Previsão perfeita")
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("Valor real (US$)")
axes[0].set_ylabel("Valor previsto (US$)")
axes[0].set_title("Real x Previsto (vermelho = fumante)")
axes[0].legend()

residuos = y_test - y_pred
sns.histplot(residuos, bins=40, kde=True, ax=axes[1], color="gray")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Distribuição dos resíduos (real − previsto)")
axes[1].set_xlabel("Resíduo (US$)")

plt.tight_layout()
plt.show()

In [ ]:
# Importância das variáveis: quanto cada uma contribuiu para reduzir o erro nas divisões
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(8, 4))
importancias.plot(kind="barh", color="teal")
plt.title("Importância das variáveis na árvore ajustada")
plt.xlabel("Importância (redução relativa do erro quadrático)")
plt.show()

importancias.sort_values(ascending=False)

---
## 5. Visualização da Árvore

Como a árvore ajustada é **rasa**, podemos plotá-la com `plot_tree` e ler diretamente as regras que ela aprendeu. Em cada nó aparecem:
- a **regra de divisão** (ex.: `smoker <= 0.5`, ou seja, "não fumante" vai para a esquerda);
- `squared_error`: o erro quadrático (variância) dos custos naquele nó;
- `samples`: quantas pessoas do treino chegaram ao nó;
- `value`: a **previsão** do nó, que é a média de `charges` dessas pessoas.

As cores mais escuras indicam previsões de custo mais altas.

In [ ]:
plt.figure(figsize=(26, 11))
plot_tree(
    modelo,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    precision=1,
    fontsize=10,
)
plt.title("Árvore de Decisão — Previsão do custo do seguro (charges)", fontsize=16)
plt.show()

In [ ]:
# As mesmas regras em formato de texto (útil quando a figura fica pequena)
print(export_text(modelo, feature_names=list(X.columns), decimals=1))

### 5.1 Árvore didática com profundidade 2

Para facilitar a leitura, treinamos também uma versão com apenas **2 níveis**. Ela é menos precisa, mas mostra as decisões mais importantes de forma bem clara.

In [ ]:
arvore_d2 = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE)
arvore_d2.fit(X_train, y_train)

print(f"R² no teste (profundidade 2): {r2_score(y_test, arvore_d2.predict(X_test)):.4f}")

plt.figure(figsize=(14, 6))
plot_tree(arvore_d2, feature_names=X.columns, filled=True, rounded=True, precision=0, fontsize=11)
plt.title("Árvore de Decisão com profundidade 2", fontsize=14)
plt.show()

---
## 6. Interpretação e Conclusão

### 6.1 O modelo teve um bom desempenho?

Sim, o desempenho foi **bom**, principalmente para um modelo tão simples e interpretável. A árvore ajustada (`max_depth=4`, `min_samples_leaf=5`, com apenas **16 folhas**) obteve **R² ≈ 0,90 no conjunto de teste**: ela explica cerca de 90% da variação do custo do seguro em dados que nunca viu. O **erro médio absoluto (MAE) ficou em torno de US$ 2,6 mil**, contra um custo médio de US$ 14,3 mil no teste. Para comparação, um modelo ingênuo que sempre prevê a média erraria cerca de US$ 9,9 mil (R² ≈ 0). O RMSE (≈ US$ 4,3 mil) é maior que o MAE porque alguns casos têm erros grandes: são principalmente **não fumantes com custos muito altos**, que aparecem como pontos isolados no gráfico Real x Previsto. Esses custos provavelmente vêm de fatores que não estão no dataset (doenças preexistentes, acidentes, internações), então nenhum modelo com essas variáveis conseguiria prevê-los bem. Também vale destacar a comparação com a **árvore sem restrições**: ela teve R² = 1,0 no treino, mas apenas ≈ 0,82 no teste. Ela "decorou" os dados, com 18 níveis e uma folha por pessoa, o que caracteriza **overfitting**. Limitar a profundidade deixou a árvore **mais simples e mais precisa** ao mesmo tempo. Por fim, o R² de teste (0,90) ficou um pouco acima do de treino (0,86), o que se deve à divisão aleatória específica dos dados. A estimativa mais realista do desempenho esperado é o **R² médio da validação cruzada, ≈ 0,83**, ainda um resultado sólido.

### 6.2 O que a árvore aprendeu (leitura do `plot_tree`)

A árvore traduz o problema em regras do tipo "se… então…", fáceis de explicar para alguém da área de negócios:

1. **A primeira e mais importante divisão é `smoker <= 0.5`** (fumante ou não). Sozinha, essa variável responde por **cerca de 70% da importância** do modelo. Na árvore de profundidade 2:
   - **Não fumantes** (855 pessoas no treino) têm custo médio de ≈ US$ 8,5 mil;
   - **Fumantes** (214 pessoas) têm custo médio de ≈ US$ 31,1 mil, quase **4 vezes mais**.

2. **Entre os não fumantes, o que mais pesa é a idade** (`age <= 42.5`). Até 42 anos, o custo médio é ≈ US$ 5,5 mil; acima disso, ≈ US$ 12,4 mil. Nos níveis seguintes, a árvore continua dividindo por faixas etárias (46, 51, 58 anos) e pelo número de filhos, formando uma "escada" em que o custo sobe com a idade, de ≈ US$ 2,6 mil (jovens até 22 anos sem filhos) a ≈ US$ 15,3 mil (acima de 58 anos).

3. **Entre os fumantes, o fator decisivo é o IMC** (`bmi <= 30.0`, que coincide exatamente com o limiar clínico de **obesidade**):
   - Fumantes com IMC ≤ 30 têm custo médio de ≈ US$ 21,2 mil;
   - Fumantes com IMC > 30 (**fumantes obesos**) têm custo médio de ≈ US$ 41,2 mil, o grupo mais caro de todos.

   Ou seja, a árvore descobriu sozinha uma **interação**: o IMC quase não importa para não fumantes, mas dobra o custo quando combinado com o tabagismo. Um modelo linear simples não capturaria isso sem que criássemos essa interação manualmente.

4. **Sexo e região praticamente não foram usados** (importância ≈ 0), o que indica que, uma vez conhecidos tabagismo, IMC e idade, essas variáveis não ajudam a prever o custo neste dataset.

**Resumo:** o modelo mostra que o custo do seguro é definido principalmente por **tabagismo → IMC → idade**, nessa ordem de importância. Com apenas 4 níveis de decisão, a árvore oferece uma boa precisão e regras transparentes, uma das grandes vantagens das árvores de decisão sobre modelos do tipo "caixa-preta". Como próximos passos, seria possível testar modelos de *ensemble* baseados em árvores (como *Random Forest* ou *Gradient Boosting*), que costumam reduzir ainda mais o erro, mas ao custo de perder essa interpretabilidade direta.